# Customer Churn Prediction

In [ ]:
import pandas as pd

accounts = pd.read_csv("ravenstack_accounts.csv")
subscriptions = pd.read_csv("ravenstack_subscriptions.csv")

feature_usage = pd.read_csv("ravenstack_feature_usage.csv")
support_tickets = pd.read_csv("ravenstack_support_tickets.csv")
churn_events = pd.read_csv("ravenstack_churn_events.csv")

In [ ]:
display(accounts.head())
accounts.info()
display(accounts.describe(include="all"))

display(subscriptions.head())
subscriptions.info()
display(subscriptions.describe(include="all"))

display(feature_usage.head())
feature_usage.info()
display(feature_usage.describe(include="all"))

display(support_tickets.head())
support_tickets.info()
display(support_tickets.describe(include="all"))

display(churn_events.head())
churn_events.info()
display(churn_events.describe(include="all"))

# Data Cleaning
### Converting to datetime



In [ ]:
accounts["signup_date"]=pd.to_datetime(accounts["signup_date"])

subscriptions["start_date"]=pd.to_datetime(subscriptions["start_date"])
subscriptions["end_date"]=pd.to_datetime(subscriptions["end_date"])

feature_usage["usage_date"]=pd.to_datetime(feature_usage["usage_date"])

support_tickets["submitted_at"]=pd.to_datetime(support_tickets["submitted_at"])
support_tickets["closed_at"]=pd.to_datetime(support_tickets["closed_at"])

churn_events["churn_date"]=pd.to_datetime(churn_events["churn_date"])

### Check missing value and investigate them

In [ ]:
print("Accounts :\n",accounts.isnull().sum())
print("\nSubscriptions :\n",subscriptions.isnull().sum())
print("\nFeature Usage :\n",feature_usage.isnull().sum())
print("\nSupport Tickets :\n",support_tickets.isnull().sum())
print("\nChurn Events :\n",churn_events.isnull().sum())

### Duplicate Rows

In [ ]:
print(accounts.duplicated().sum())
print(subscriptions.duplicated().sum())
print(feature_usage.duplicated().sum())
print(support_tickets.duplicated().sum())
print(churn_events.duplicated().sum())

### Basic Sanity Check

In [ ]:
print(subscriptions["seats"].describe())
print(subscriptions["mrr_amount"].describe())
print(subscriptions["arr_amount"].describe())
print(feature_usage["usage_count"].describe())
print(support_tickets["resolution_time_hours"].describe())
print(support_tickets["first_response_time_minutes"].describe())
print(churn_events["refund_amount_usd"].describe())

# Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

### Chart 1 - Plan Distribution

In [ ]:
# Customer distribution by plan tier
plt.figure(figsize=(4, 3))
accounts["plan_tier"].value_counts().plot(kind="bar")
plt.title("Customer distribution by plan")
plt.xlabel("Plan Tier")
plt.ylabel("Number of Accounts")
plt.show()

### Chart 2 - Industry Distribution

In [ ]:
# Customer Distribution by industury
plt.figure(figsize=(4,3))
accounts["industry"].value_counts().plot(kind="bar")
plt.title("Customer distibution by industry")
plt.show()

### Chart 3 - MRR Distribution & ARR Distribution

In [ ]:
# Histplot for monthly recurring revenue distribution & annual recurring revenue
plt.figure(figsize=(13,5))
sns.histplot(subscriptions["mrr_amount"],bins=100)
plt.title("MRR Distribution")
plt.show()
plt.figure(figsize=(13,5))
sns.histplot(subscriptions["arr_amount"],bins=100)
plt.title("ARR Distribution")
plt.show()

### Chart 4 - Country Distribution

In [ ]:
# Customer distribution by country
plt.figure(figsize=(4, 3))
accounts["country"].value_counts().plot(kind="bar")
plt.title("Customer distribution by Country")
plt.xlabel("Country")
plt.ylabel("Number of Customers")
plt.show()

### Chart 5 - Seats Distribution

In [ ]:
plt.figure(figsize=(13,4))
sns.boxplot(subscriptions["seats"],orient="h")
plt.title("Seats Distribution")
plt.show()

### Chart 6 - Referral-Source distribution

In [ ]:
# Customer distribution by referral-source
plt.figure(figsize=(4, 3))
accounts["referral_source"].value_counts().plot(kind="bar")
plt.title("Customer distribution by Referral Source")
plt.xlabel("Referral Source")
plt.ylabel("Number of Customers")
plt.show()

### Chart 7 - Reason Code Distribution 

In [ ]:
# Reason code distribution
# What is the main reason churn
plt.figure(figsize=(4, 3))
churn_events["reason_code"].value_counts().plot(kind="bar")
plt.title("Churn Distribution by reason code")
plt.xlabel("Reason code")
plt.ylabel("Number of Customers who churn")
plt.show()

# Trend Analysis

### Monthly Trend Analysis

In [ ]:
plt.figure(figsize=(13,4))
subscriptions["month"]=subscriptions["start_date"].dt.to_period("M")
monthly_revenue = subscriptions.groupby("month")["mrr_amount"].sum()

monthly_revenue.plot(color="Green",linestyle="dashed",linewidth="3")
plt.title("Monthly Revenue Trend")
plt.ylabel("Total MRR")
plt.show()

### Monthly New Subscriptions Trend

In [ ]:
plt.figure(figsize=(13,4))
monthly_subs = subscriptions.groupby("month")["subscription_id"].count()
monthly_subs.plot(color="Green",marker="*",markersize="10")
plt.title("Monthly new Subcriptions")
plt.show()

### Monthly Churn Trend

In [ ]:
plt.figure(figsize=(13,4))
churn_events['churn_month'] = churn_events['churn_date'].dt.to_period('M')
monthly_churn = churn_events.groupby('churn_month')['churn_event_id'].count()
monthly_churn.plot(color='red')
plt.title("Monthly Churn Events")
plt.show()

# KPI Engineering

### Subscription Aggregation

In [ ]:
subscription_span = subscriptions.copy()

max_dataset_date = max(
    subscriptions['start_date'].max(),
    churn_events['churn_date'].max()
)

subscription_span['end_date'] = subscription_span['end_date'].fillna(max_dataset_date)

subscription_span_agg = subscription_span.groupby('account_id').agg(
    first_subscription=('start_date', 'min'),
    last_subscription=('end_date', 'max')
).reset_index()

subscription_span_agg['active_days'] = (
    subscription_span_agg['last_subscription']
    - subscription_span_agg['first_subscription']
).dt.days

# creating Revenue Aggregation
subscription_revenue = subscriptions.groupby("account_id").agg(
    total_mrr=("mrr_amount", "sum"),
    total_arr=("arr_amount", "sum"),
    avg_mrr=("mrr_amount", "mean"),
    avg_seats=("seats", "mean")
).reset_index()

# Merge Revenue + active days = Final aggregated subscription  
subscription_agg = subscription_revenue.merge(
    subscription_span_agg[['account_id', 'active_days']],
    on='account_id',
    how='left'
)
# check
display(subscription_agg.head())
display(subscription_agg.describe())


### Ussage Aggregation

In [ ]:
# merge usage with subscription for account_id
usage_with_account = feature_usage.merge(
    subscriptions[["subscription_id","account_id"]],
    on="subscription_id",
    how="left"
)
# check
#display(usage_with_account)
#print(usage_with_account["account_id"].isnull().sum())

# now aggregate
usage_agg=usage_with_account.groupby("account_id").agg(
    total_usage=("usage_count","sum"),
    avg_usage=("usage_count","mean")
).reset_index()
# check
display(usage_agg.head())

### Support Ticket Aggregation

In [ ]:
ticket_agg=support_tickets.groupby("account_id").agg(
    total_tickets=("ticket_id","count"),
    avg_resolution_time_hours=("resolution_time_hours","mean"),
    avg_first_response_time_minutes=("first_response_time_minutes","mean")
).reset_index()
display(ticket_agg.head())

### Churn flag

In [ ]:
churn_flag = churn_events[['account_id']].drop_duplicates()
churn_flag['churn_flag'] = 1
display(churn_flag.head())
display(churn_events.describe(include="all"))

## Merge Everything in account Kpi summary table

In [ ]:
# Exctracting only necessary columns
account_kpi = accounts[[
    'account_id',
    'account_name',
    'industry',
    'country',
    'signup_date',
    'referral_source',
    'plan_tier'
]].copy()

account_kpi=account_kpi.merge(subscription_agg,on="account_id",how="left")
account_kpi=account_kpi.merge(usage_agg,on="account_id",how="left")
account_kpi=account_kpi.merge(ticket_agg,on="account_id",how="left")
account_kpi=account_kpi.merge(churn_flag,on="account_id",how="left")
display(account_kpi.head(20))
display(account_kpi.describe())

### Data Cleaning after merge

In [ ]:
# fill churn nulls
account_kpi["churn_flag"]=account_kpi["churn_flag"].fillna(0)

# usage and ticket nulls (accounts with no activity)
account_kpi[[
    "total_usage","avg_usage","total_tickets","avg_resolution_time_hours","avg_first_response_time_minutes"
]]=account_kpi[[
    "total_usage","avg_usage","total_tickets","avg_resolution_time_hours","avg_first_response_time_minutes"
]].fillna(0)
display(account_kpi.describe())

In [ ]:
# Data type Correction
# before data type correction
display(account_kpi.info())
account_kpi[[
    "total_tickets","avg_resolution_time_hours","avg_first_response_time_minutes","churn_flag"
]]=account_kpi[[
    "total_tickets","avg_resolution_time_hours","avg_first_response_time_minutes","churn_flag"
]].astype(int)
# after data type correction
display(account_kpi.info())

### Exporting the account_kpi to CSV

In [ ]:
account_kpi.to_csv("account_kpi.csv", index=False)
# in excel
account_kpi.to_excel("account_kpi.xlsx",index=False)

# KPI & Insights

### Core Metrics

In [ ]:
print("Total_account: ",account_kpi.shape[0])
print("Total_churn: ",account_kpi['churn_flag'].sum())
print("Churn_rate: ",account_kpi['churn_flag'].mean()*100)
print("Total Revenue: ",account_kpi['total_mrr'].sum())
arpu = account_kpi["total_mrr"].sum() / account_kpi.shape[0]
print("Average revenue per user: ",arpu)

revenue_lost = account_kpi.loc[account_kpi["churn_flag"]==1,"total_mrr"].sum()
print("Revenue Lost:", revenue_lost)

revenue_lost_pct = revenue_lost / account_kpi["total_mrr"].sum()
print("Revenue Lost %:", revenue_lost_pct*100)

monthly_revenue = account_kpi["avg_mrr"].sum()
print("Average Monthly Revenue: ",monthly_revenue)

monthly_revenue_lost = account_kpi.loc[account_kpi["churn_flag"]==1, "avg_mrr"].sum()
print("Average Monthly Revenue lost: ", monthly_revenue_lost)

### Segement Analysis

In [ ]:
churn_by_plan = account_kpi.groupby("plan_tier")["churn_flag"].mean()
print("Churn % by plan:\n",churn_by_plan*100)

revenue_by_plan = account_kpi.groupby("plan_tier")["total_mrr"].sum()
print("Total Revenue by plan:\n",revenue_by_plan)

revenue_lost_by_plan = account_kpi[account_kpi["churn_flag"]==1].groupby("plan_tier")["total_mrr"].sum()
print("Total Revenue lost by plan:\n",revenue_lost_by_plan)

### Behavioural Analysis

In [ ]:
usage_churn = account_kpi.groupby(
    pd.qcut(account_kpi["total_usage"],3)
)["churn_flag"].mean()
print("Churn rate based on usage:\n",usage_churn*100)

ticket_churn = account_kpi.groupby(
    pd.qcut(account_kpi["total_tickets"],3)
)["churn_flag"].mean()
print("Churn rate based on tickets raised:\n",ticket_churn*100)

### Country Analysis

In [ ]:
churn_by_country = account_kpi.groupby("country")["churn_flag"].mean()
print("Churn rate by country:\n",churn_by_country*100)

revenue_by_country = account_kpi.groupby("country")["total_mrr"].sum().sort_values(ascending=False)
print("Revenue by country:\n",revenue_by_country)